#### Dropout
: 오버피팅이 발생했을 때 학습 데이터의 10%~30% 정도를 비활성하고 학습을 시도함

In [16]:
from multiclass_functions import *

In [2]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import numpy as np
import os
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
dropout = nn.Dropout(p=0.3)     # p : 각 노드를 비활성화할 확률 | 전체 비율의 30%라는 의미 X
x = torch.randint(1,101,(10,)).float()

print('원본 :', x)
print('원본 합계 :', x.sum())

원본 : tensor([ 28.,  97.,  98.,  88., 100.,  86.,  65.,  86.,  21.,  20.])
원본 합계 : tensor(689.)


In [ ]:
# 학습모드일 떄 dropout 활성화
dropout.train()
train1 = dropout(x)
print('Train 1:', train1)       # dropout되지 않은 값들은 1-p로 나눈 값을 출력 | 연산 결과를 원본과 비슷하게 유지
print('Train 1 SUM', train1.sum())

dropout.train()
train2 = dropout(x)
print('Train 2:', train2)
print('Train 2 SUM', train2.sum())

dropout.train()
train3 = dropout(x)
print('Train 3:', train3)
print('Train 3 SUM', train3.sum())
print('-'*50)

Train 1: tensor([ 40.0000, 138.5714,   0.0000,   0.0000, 142.8571, 122.8571,  92.8571,
        122.8571,  30.0000,  28.5714])
Train 1 SUM tensor(718.5714)
Train 2: tensor([ 40.0000, 138.5714,   0.0000, 125.7143,   0.0000, 122.8571,  92.8571,
        122.8571,  30.0000,   0.0000])
Train 2 SUM tensor(672.8572)
Train 3: tensor([ 40.0000, 138.5714,   0.0000,   0.0000, 142.8571, 122.8571,  92.8571,
        122.8571,  30.0000,  28.5714])
Train 3 SUM tensor(718.5714)
--------------------------------------------------


In [ ]:
# 평가모드 일 때 dropout 비활성화
dropout.eval()
print('Eval :', dropout(x)) # 원본 그대로 유지

# 학습 모듈을 epoch를 돌면서 학습하고 평가까지 한꺼번에 진행하는 경우 학습이 아닌 평가에서는 dropout할 필요 없기에 넣은 기능

Eval : tensor([ 28.,  97.,  98.,  88., 100.,  86.,  65.,  86.,  21.,  20.])


In [ ]:
class sample_model(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.drop_layer = nn.Sequential(
            nn.Linear(5,7),
            nn.Tanh(),
            nn.Dropout(p=.2)
            )
    def forward(self, x, *args, **kwargs):
        x = self.drop_layer(x)
        return x

model = sample_model()
model.train()
x = torch.randn(2,3,5)
print(x)
print(model(x))

print()

model.eval()
print(model(x))

tensor([[[-0.9980,  0.5529, -1.2768,  0.2616, -1.0579],
         [-0.7606,  0.5813,  0.6863, -0.7433,  1.1426],
         [-1.3952, -0.4265, -1.1569, -0.0523,  0.0651]],

        [[-1.0854,  1.8846, -2.4032,  1.2265,  0.4752],
         [ 0.4798, -1.1871, -0.8773,  1.4725, -0.6736],
         [ 0.4721, -0.4459,  0.5253, -0.9378,  2.2012]]])
tensor([[[ 0.9414, -0.0000, -0.8907,  1.1270, -0.3563,  0.4091, -0.0000],
         [ 0.6322, -0.0000,  0.1306, -0.2307, -0.3621, -0.4364,  0.5029],
         [ 1.0250, -0.1409, -0.0000,  0.6496, -0.2787,  0.0000,  0.2042]],

        [[ 0.0000,  0.0000, -0.7554,  1.1760,  0.9988,  0.8329, -0.3468],
         [ 0.2200, -0.3573, -0.4863,  0.0000, -0.4835,  0.0000,  0.2638],
         [ 0.0610, -0.0000,  0.0000, -0.9378,  0.0638,  0.0936,  0.9257]]],
       grad_fn=<MulBackward0>)

tensor([[[ 0.7531, -0.0651, -0.7126,  0.9016, -0.2851,  0.3273, -0.1564],
         [ 0.5057, -0.3419,  0.1045, -0.1846, -0.2897, -0.3491,  0.4024],
         [ 0.8200, -0.1127, -0.6

In [ ]:
class sample_model_CNN(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.drop_layer = nn.Sequential(
            nn.Conv2d(3,3,3),
            nn.Dropout2d(p=.2)  # 2d 일 경우 행렬 하나가 통째로 비활성화
            )
    def forward(self, x, *args, **kwargs):
        x = self.drop_layer(x)
        return x

model = sample_model_CNN()
model.train()

x = torch.randn(5,3,4,4)
print(x)
print(model(x))

print()

model.eval()
print(model(x))

tensor([[[[ 7.7308e-01,  8.9943e-02,  5.8521e-01,  6.5185e-04],
          [-9.6752e-01,  2.8972e-01,  6.5429e-01, -7.2681e-01],
          [-7.5169e-01, -1.7061e+00, -1.6318e+00,  4.1730e-01],
          [-7.9181e-02,  1.6003e+00, -1.2149e+00, -3.7471e-01]],

         [[ 5.3695e-01,  2.2529e-02,  2.2648e-01, -2.4684e+00],
          [ 1.1817e-01, -1.0830e-03, -3.8084e-01,  1.5197e-01],
          [ 1.3049e+00,  9.8847e-01, -5.2379e-01,  1.8112e+00],
          [ 2.6275e-01,  2.5432e+00,  1.0854e+00, -4.4912e-01]],

         [[ 2.3554e-01, -1.5508e+00,  1.0758e+00, -5.4968e-01],
          [-9.4668e-01,  6.9347e-01, -1.3638e+00, -2.5012e-02],
          [-2.7622e-01, -1.4418e+00, -1.1521e+00,  2.3169e-01],
          [ 8.4194e-01, -7.4370e-01,  1.9739e+00, -7.1064e-01]]],


        [[[-2.6200e+00,  1.3073e+00,  1.4462e+00,  7.7760e-01],
          [-9.7260e-01, -1.1010e+00,  1.1829e+00, -7.0952e-01],
          [-4.0994e-01,  8.3616e-01, -8.5692e-01, -9.1273e-01],
          [ 1.4676e+00, -6.8872e